# Pokémon Battle Prediction 2026 — Competitive XGBoost Pipeline
**Sapienza University of Rome — Foundations of Data Science Competition**

This notebook implements our end-to-end, reproducible, competition-compliant **XGBoost** solution for predicting the final winner (`player_won` $\in \{0, 1\}$) of Pokémon Generation 2 OU battles from incomplete 30-turn battle observations (`battle_timeline`).

### Key Methodological Highlights
1. **Strict Competition Compliance**: Uses only classical gradient-boosted decision trees (`XGBClassifier`), the provided competition JSONL datasets, and the official Generation 2 type chart (`type_chart_gen2.csv`). No neural networks, pretrained models, or external lookup tables are used.
2. **Gen 2 Battle State Reconstruction (`src/battle_features.py`)**:
   - **Stateful Move-Cache De-duplication**: Filters out voluntary switch-in turns where the dataset parser repeats a Pokémon's previously cached `move_details`, preventing severe double-counting of offensive moves and status actions.
   - **Player 2 Single-KO Physics Reconstruction**: Because `p2_pokemon_state.status == "fnt"` only appears on simultaneous double-KOs and Turn-30 KOs (when Player 2's Pokémon faints alone on Turns 1–29, its replacement immediately appears in the turn snapshot), we reconstruct P2 single-KOs using Gen 2 physics (`y_untouched` replacement verification, P1 attack execution, type effectiveness, residual KO tracking, and Explosion self-KOs).
   - **Within-Fold Species Catalogue**: Across all battles, species `name` uniquely determines `(types, base_hp, base_atk, base_def, base_spa, base_spd, base_spe)` with zero inconsistencies. We construct this catalogue strictly from the training data to infer stats and Gen 2 type matchups for Player 2's revealed Pokémon.
3. **Six Modular Feature Families + Leak-Free Within-Fold Target Encoding**:
   - **Family A (`team`)**: Player 1 full-team stat distributions, physical/special bulk, speed control, type diversity, and Gen 2 OU meta staples.
   - **Family B (`p2_observed`)**: Player 2 lead & revealed roster stats, unrevealed reserve uncertainty (using `NaN` rather than `0` for unobserved quantities), and observed move categories.
   - **Family C (`state`)**: Turn-30 material balance, alive/fainted differentials, recency-decayed HP pools, weighted status burden, active stat boosts, and phazed-absence scores.
   - **Family D (`temporal`)**: Multi-window (`Turns 1–10`, `11–20`, `21–30`, `Last 5`, `Last 10`) HP trajectories, slope/momentum, lead changes, recovery events, and tempo control.
   - **Family E (`matchup`)**: Gen 2 type effectiveness matrices (`type_chart_gen2.csv`), active STAB multipliers, best available roster counter multipliers, and Speed/KO threat indices.
   - **Family F (`interactions`)**: Non-linear domain ratios, effective HP-adjusted survival advantage, and Turn-30 active sweep threat indices.
   - **Strictly Leak-Free `WithinFoldTargetEncoder`**: Computes smoothed empirical-Bayes species, lead-matchup, and HP-weighted alive-species win rates strictly inside each CV fold using inner 5-fold out-of-fold splits.

## Step 1 & 2: Environment Setup and GitHub Repository Cloning
When running inside Kaggle Notebooks, this cell clones the project repository and adds it to `sys.path`. When running locally inside the repository clone, it automatically detects the local workspace without re-cloning.

In [1]:
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = os.environ.get("REPO_URL", "https://github.com/danielegabriele17-rgb/predictive-spikes.git")
GIT_REF = os.environ.get("GIT_REF", "main")
CLONE_DIR = Path("/kaggle/working/predictive-spikes") if Path("/kaggle/working").exists() else Path.cwd()

def setup_repository() -> Path:
    # Case 1: Already running inside the repository (local execution or pre-cloned workspace)
    for candidate in [Path.cwd(), Path.cwd().parent, CLONE_DIR]:
        if (candidate / "src" / "battle_features.py").exists() and (candidate / "configs" / "experiments.yaml").exists():
            print(f"Using existing repository at: {candidate.resolve()}")
            return candidate.resolve()

    # Case 2: Clone from GitHub into /kaggle/working/predictive-spikes
    print(f"Cloning {REPO_URL} (ref={GIT_REF}) into {CLONE_DIR}...")
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_REF, REPO_URL, str(CLONE_DIR)], check=True)
    return CLONE_DIR.resolve()

REPO_ROOT = setup_repository()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else (REPO_ROOT / "outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository Root : {REPO_ROOT}")
print(f"Output Directory: {OUTPUT_DIR}")

Using existing repository at: /Users/Daniele/Documents/GitHub/predictive-spikes
Repository Root : /Users/Daniele/Documents/GitHub/predictive-spikes
Output Directory: /Users/Daniele/Documents/GitHub/predictive-spikes/outputs


## Step 3: Import Modular Pipeline Components (`src/`)

In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

from src.battle_features import build_species_catalogue
from src.data_loader import discover_data_dir, load_competition_data
from src.feature_engineering import (
    ALL_FEATURE_FAMILIES,
    WithinFoldTargetEncoder,
    build_feature_dataframe,
)
from src.model import get_feature_importance_df, run_xgb_cv
from src.submission import generate_submission, validate_submission_df
from src.validation import get_cv_splits

print(f"XGBoost version: {xgb.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")

XGBoost version: 3.4.1
NumPy version  : 2.5.4
Pandas version : 3.1.0


## Step 4 & 5: Competition Dataset Discovery and Loading
The environment-aware loader (`discover_data_dir` and `load_competition_data`) automatically locates `train.jsonl`, `test.jsonl`, `sample_submission.csv`, and `type_chart_gen2.csv` whether running in Kaggle (`/kaggle/input/...`) or locally (`fds-pokemon-battles-prediction-2026/`).

In [3]:
t_start = time.perf_counter()
data_dir = discover_data_dir()
train_battles, test_battles, sample_sub, type_chart = load_competition_data(custom_dir=data_dir)
y = np.array([int(b["player_won"]) for b in train_battles], dtype=int)

print(f"Discovered competition data directory: {data_dir}")
print(f"Training battles : {len(train_battles):,} | Target positive rate: {y.mean():.4f}")
print(f"Test battles     : {len(test_battles):,} | Sample submission shape: {sample_sub.shape}")
print(f"Gen 2 Type Chart : {len(type_chart)} attack-defense type pairs loaded")

Discovered competition data directory: /Users/Daniele/Documents/GitHub/predictive-spikes/fds-pokemon-battles-prediction-2026
Training battles : 10,000 | Target positive rate: 0.5000
Test battles     : 5,000 | Sample submission shape: (5000, 2)
Gen 2 Type Chart : 110 attack-defense type pairs loaded


## Step 6: Feature Engineering and Leak-Free Within-Fold Target Encoding
We extract all 6 modular feature families (`team`, `p2_observed`, `state`, `temporal`, `matchup`, `interactions`) across the 10,000 training battles and 5,000 test battles, and then compute strictly out-of-fold empirical-Bayes target encodings inside each 5-fold `StratifiedKFold` split.

In [4]:
t_fe = time.perf_counter()
catalogue = build_species_catalogue(train_battles)
print(f"Built internal species stat/type catalogue from training set: {len(catalogue)} unique species")

X_train_full, train_ids, y_extracted = build_feature_dataframe(
    train_battles,
    families=ALL_FEATURE_FAMILIES,
    catalogue=catalogue,
    type_chart=type_chart,
)
X_test_full, test_ids, _ = build_feature_dataframe(
    test_battles,
    families=ALL_FEATURE_FAMILIES,
    catalogue=catalogue,
    type_chart=type_chart,
)
print(f"Extracted feature matrices in {time.perf_counter() - t_fe:.2f}s:")
print(f"  X_train_full: {X_train_full.shape}")
print(f"  X_test_full : {X_test_full.shape}")

Built internal species stat/type catalogue from training set: 239 unique species
Extracted feature matrices in 9.78s:
  X_train_full: (10000, 549)
  X_test_full : (5000, 549)


## Step 7: Load Version-Controlled Experiment & Model Configurations (`configs/experiments.yaml`)
All selected feature subsets (`top140_features`, `top200_features`, `top220_features`, `pruned_features`), Optuna-tuned and regularized hyperparameters, and ensemble weights are loaded from `configs/experiments.yaml` for 100% reproducibility.

In [5]:
config_path = REPO_ROOT / "configs" / "experiments.yaml"
with open(config_path, "r", encoding="utf-8") as f:
    exp_config = yaml.safe_load(f)

seed = int(exp_config.get("random_seed", 42))
n_splits = int(exp_config.get("n_splits", 5))
threshold = float(exp_config.get("default_threshold", 0.5))
feature_subsets = exp_config["feature_subsets"]
models_cfg = exp_config["models"]
ens_cfg = exp_config["ensemble"]

print(f"Loaded configuration from: {config_path}")
for subset_name, cols in feature_subsets.items():
    print(f"  Feature subset [{subset_name:18s}]: {len(cols)} features")
print("Configured models:", list(models_cfg.keys()))

Loaded configuration from: /Users/Daniele/Documents/GitHub/predictive-spikes/configs/experiments.yaml
  Feature subset [pruned_features   ]: 511 features
  Feature subset [top200_features   ]: 200 features
  Feature subset [top220_features   ]: 220 features
  Feature subset [top140_features   ]: 140 features
Configured models: ['model1_optuna_top200_te', 'model2_conservative_depth4_pruned_te', 'model3_obs_only_top220_no_te', 'model4_compact_top140_te']


## Step 8 & 9: Stratified 5-Fold Cross-Validation, Within-Fold Encoding, and Model Fitting
We construct the canonical 5-fold `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` splits, fit `WithinFoldTargetEncoder` strictly on each fold's training split (using nested inner 5-fold OOF encodings for the training rows), and train all four regularized XGBoost model families with fold-level early stopping.

In [6]:
cv_splits = get_cv_splits(train_battles, y, n_splits=n_splits, seed=seed, group_duplicates=False)

te_folds_train = []
te_folds_val = []
te_folds_test = []
for tr_idx, va_idx in cv_splits:
    enc = WithinFoldTargetEncoder(smoothing=30.0, team_smoothing=20.0)
    tr_b = [train_battles[i] for i in tr_idx]
    va_b = [train_battles[i] for i in va_idx]
    te_folds_train.append(enc.fit_transform_inner_oof(tr_b, y[tr_idx], n_inner_splits=5, seed=seed))
    te_folds_val.append(enc.transform(va_b))
    te_folds_test.append(enc.transform(test_battles))

fold_map = {(int(tr[0]), int(va[0])): idx for idx, (tr, va) in enumerate(cv_splits)}

def fold_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_train[idx], te_folds_val[idx]

def test_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_test[idx]

oof_probs_map = {}
test_probs_map = {}
cv_results_map = {}

for m_name, m_spec in models_cfg.items():
    cols = feature_subsets[m_spec["feature_subset"]]
    use_te = bool(m_spec.get("use_within_fold_te", True))
    es_rounds = m_spec.get("early_stopping_rounds", 50)
    params = m_spec["xgb_params"]

    cv_res, test_probs = run_xgb_cv(
        X_df=X_train_full[cols],
        y=y,
        splits=cv_splits,
        xgb_params=params,
        experiment_id=m_name,
        early_stopping_rounds=es_rounds,
        fold_feature_fn=fold_te_fn if use_te else None,
        X_test_df=X_test_full[cols],
        test_fold_feature_fn=test_te_fn if use_te else None,
        threshold=threshold,
        seed=seed,
    )
    cv_results_map[m_name] = cv_res
    oof_probs_map[m_name] = cv_res.oof_probs
    test_probs_map[m_name] = test_probs
    print(
        f"[{m_name:36s}] 5-Fold CV Acc: {cv_res.cv_mean_acc:.4f} +/- {cv_res.cv_std_acc:.4f} "
        f"| ROC-AUC: {cv_res.oof_roc_auc:.4f} | LogLoss: {cv_res.oof_log_loss:.4f} "
        f"| Train-Val Gap: {np.mean(cv_res.fold_train_accs) - cv_res.cv_mean_acc:.4f}"
    )

[model1_optuna_top200_te             ] 5-Fold CV Acc: 0.6970 +/- 0.0022 | ROC-AUC: 0.7738 | LogLoss: 0.5662 | Train-Val Gap: 0.0916
[model2_conservative_depth4_pruned_te] 5-Fold CV Acc: 0.7008 +/- 0.0024 | ROC-AUC: 0.7748 | LogLoss: 0.5657 | Train-Val Gap: 0.1072
[model3_obs_only_top220_no_te        ] 5-Fold CV Acc: 0.6988 +/- 0.0046 | ROC-AUC: 0.7714 | LogLoss: 0.5692 | Train-Val Gap: 0.1453
[model4_compact_top140_te            ] 5-Fold CV Acc: 0.7020 +/- 0.0036 | ROC-AUC: 0.7752 | LogLoss: 0.5647 | Train-Val Gap: 0.1286


### Top Predictive Features Inspection
Let us inspect the top 20 most informative features by gain from our primary single model (`model4_compact_top140_te`).

In [7]:
cv_m4 = cv_results_map["model4_compact_top140_te"]
imp_df = get_feature_importance_df(cv_m4.models, cv_m4.feature_names)
print(imp_df.head(20).to_string(index=False))

                      feature  importance
inter_combined_material_score    0.065198
          state_hp_ratio_soft    0.031119
           state_hp_diff_soft    0.029498
           state_hp_ratio_adj    0.022312
          temp_hp_adv_adj_t30    0.022255
   temp_last5_hp_adv_adj_mean    0.019759
            state_hp_diff_raw    0.013343
          state_clean_hp_diff    0.013288
            state_spikes_diff    0.011726
      inter_hp_raw_x_fnt_diff    0.011374
  inter_status_x_remaining_hp    0.011199
   inter_projected_hp_adv_t40    0.010487
           state_hp_ratio_raw    0.010266
       state_p1_spikes_active    0.010174
 inter_spikes_x_grounded_diff    0.009161
        state_alive_diff_soft    0.008891
         state_boost_sum_diff    0.008521
     inter_hp_adv_x_alive_adv    0.008394
           te_p1_team_winrate    0.007525
           state_p1_boost_sum    0.007202


---
## Step 10, 11 & 12: Generating and Validating the Three Final Submissions

Per the competition guidelines, we produce **three distinct, high-performing final submissions** designed to balance single-model sharpness, structural regularization against leaderboard shakeup, and multi-configuration variance reduction.

---
### Final Submission 1: Compact Top-140 Feature Subset + Within-Fold Target Encoding (`submission_1_compact_top140_xgb.csv`)
**Methodology & Rationale**:
- **Architecture**: Fold-averaged ensemble of 5 `XGBClassifier` models (`max_depth=5`, `learning_rate=0.032`, `min_child_weight=10`, `subsample=0.80`, `colsample_bytree=0.60`, `colsample_bylevel=0.80`, `gamma=0.2`, `reg_alpha=0.5`, `reg_lambda=2.5`) trained with early stopping (`early_stopping_rounds=50` on validation log-loss).
- **Feature Representation (152 total features)**: Selects the top 140 collinearity-pruned engineered features (`|r| <= 0.95`) ranked by gain across all 6 domain families (`team`, `p2_observed`, `state`, `temporal`, `matchup`, `interactions`), augmented with 12 leak-free within-fold empirical-Bayes target-encoded features (including HP-weighted surviving species win-rate expectations and P1 team signature priors).
- **Why Selected**: Removing lower-gain tail features reduces tree split dilution under column subsampling, yielding our highest single-model 5-fold cross-validation accuracy (**`0.7020 +/- 0.0036`**, OOF ROC-AUC **`0.7752`**, LogLoss **`0.5647`**).

In [8]:
sub1_path = OUTPUT_DIR / "submission_1_compact_top140_xgb.csv"
sub1_preds = (test_probs_map["model4_compact_top140_te"] >= threshold).astype(int)
sub1_df = generate_submission(test_ids, sub1_preds, sample_sub, output_path=sub1_path)
sub1_val = validate_submission_df(sub1_df, sample_sub)
print(f"Saved Submission 1 -> {sub1_path}")
print("Validation summary:", sub1_val)
sub1_df.head()

Saved Submission 1 -> /Users/Daniele/Documents/GitHub/predictive-spikes/outputs/submission_1_compact_top140_xgb.csv
Validation summary: {'valid': True, 'n_rows': 5000, 'positive_count': 2601, 'negative_count': 2399, 'positive_rate': 0.5202}


---
### Final Submission 2: Conservative Regularized Depth-4 Full-Spectrum XGBoost (`submission_2_conservative_depth4_xgb.csv`)
**Methodology & Rationale**:
- **Architecture**: Shallow, heavily regularized `XGBClassifier` (`max_depth=4`, `learning_rate=0.03`, `min_child_weight=15`, `subsample=0.75`, `colsample_bytree=0.45`, `colsample_bylevel=0.75`, `gamma=0.4`, `reg_alpha=1.5`, `reg_lambda=5.0`) trained across all 5 stratified folds.
- **Feature Representation (523 total features)**: Retains the full collinearity-pruned feature space (512 engineered features across all 6 domain families + 11 within-fold target-encoded features), allowing shallow depth-4 trees with aggressive column subsampling (`colsample_bytree=0.45`) to harvest subtle secondary signals from temporal windows and matchup matrices without memorizing high-order feature combinations.
- **Why Selected**: Achieves **`0.7008 +/- 0.0024`** 5-fold CV Accuracy (OOF ROC-AUC **`0.7748`**, LogLoss **`0.5657`**) with a smaller train-validation gap (`0.1072`) and lower fold variance (`+/- 0.0024`), providing strong protection against private leaderboard distribution shift.

In [9]:
sub2_path = OUTPUT_DIR / "submission_2_conservative_depth4_xgb.csv"
sub2_preds = (test_probs_map["model2_conservative_depth4_pruned_te"] >= threshold).astype(int)
sub2_df = generate_submission(test_ids, sub2_preds, sample_sub, output_path=sub2_path)
sub2_val = validate_submission_df(sub2_df, sample_sub)
print(f"Saved Submission 2 -> {sub2_path}")
print("Validation summary:", sub2_val)
sub2_df.head()

Saved Submission 2 -> /Users/Daniele/Documents/GitHub/predictive-spikes/outputs/submission_2_conservative_depth4_xgb.csv
Validation summary: {'valid': True, 'n_rows': 5000, 'positive_count': 2597, 'negative_count': 2403, 'positive_rate': 0.5194}


---
### Final Submission 3 (Primary Default Submission): Diverse Multi-Configuration XGBoost Probability Blend (`submission_3_diverse_xgb_ensemble.csv` & `submission.csv`)
**Methodology & Rationale**:
- **Architecture**: Weighted out-of-fold and test probability blend (`weights = [0.15, 0.35, 0.10, 0.40]`) combining four structurally distinct XGBoost configurations:
  1. `model1_optuna_top200_te` (`weight = 0.15`): Optuna TPE-tuned `max_depth=4, min_child_weight=20` model on Top-200 features + Within-Fold TE (lowest train-val gap `0.0916`, CV Acc `0.6970`).
  2. `model2_conservative_depth4_pruned_te` (`weight = 0.35`): Shallow `max_depth=4, reg_lambda=5.0` model on all 512 collinearity-pruned features + Within-Fold TE (CV Acc `0.7008`).
  3. `model3_obs_only_top220_no_te` (`weight = 0.10`): **Pure Observation-Only (Zero Target Encoding)** `max_depth=5` model on Top-220 features (CV Acc `0.6988`), acting as a domain-physics anchor completely independent of species target priors.
  4. `model4_compact_top140_te` (`weight = 0.40`): High-signal Compact Top-140 + Within-Fold TE `max_depth=5` model (CV Acc `0.7020`).
- **Why Selected**: Blending across distinct tree depths (`4` vs `5`), feature dimensionalities (`140`, `200`, `220`, `512`), and target-encoding regimes (with TE vs. pure observation-only) outperforms every individual constituent on all metrics: **5-Fold CV Accuracy = `0.7030 +/- 0.0017`**, **OOF ROC-AUC = `0.7767`**, and **OOF LogLoss = `0.5636`**, while cutting fold standard deviation down to `0.0017`. This ensemble is also saved as the default `submission.csv`.

In [10]:
members = ens_cfg["members"]
weights = np.asarray(ens_cfg["weights"], dtype=float)
weights = weights / np.sum(weights)

ens_oof_probs = sum(w * oof_probs_map[m] for w, m in zip(weights, members))
ens_test_probs = sum(w * test_probs_map[m] for w, m in zip(weights, members))

ens_fold_accs = [
    float(accuracy_score(y[va_idx], (ens_oof_probs[va_idx] >= threshold).astype(int)))
    for _, va_idx in cv_splits
]
print(
    f"[Ensemble Probability Blend] 5-Fold CV Acc: {np.mean(ens_fold_accs):.4f} +/- {np.std(ens_fold_accs):.4f} "
    f"| ROC-AUC: {roc_auc_score(y, ens_oof_probs):.4f} | LogLoss: {log_loss(y, ens_oof_probs):.4f}"
)

sub3_path = OUTPUT_DIR / "submission_3_diverse_xgb_ensemble.csv"
default_sub_path = OUTPUT_DIR / "submission.csv"
sub3_preds = (ens_test_probs >= threshold).astype(int)

sub3_df = generate_submission(test_ids, sub3_preds, sample_sub, output_path=sub3_path)
generate_submission(test_ids, sub3_preds, sample_sub, output_path=default_sub_path)
sub3_val = validate_submission_df(sub3_df, sample_sub)

print(f"Saved Submission 3 -> {sub3_path}")
print(f"Saved Default Sub  -> {default_sub_path}")
print("Validation summary:", sub3_val)
sub3_df.head()

[Ensemble Probability Blend] 5-Fold CV Acc: 0.7030 +/- 0.0017 | ROC-AUC: 0.7767 | LogLoss: 0.5636
Saved Submission 3 -> /Users/Daniele/Documents/GitHub/predictive-spikes/outputs/submission_3_diverse_xgb_ensemble.csv
Saved Default Sub  -> /Users/Daniele/Documents/GitHub/predictive-spikes/outputs/submission.csv
Validation summary: {'valid': True, 'n_rows': 5000, 'positive_count': 2604, 'negative_count': 2396, 'positive_rate': 0.5208}


## Final Verification & Pairwise Agreement Check
We verify that all generated submission files strictly satisfy all competition constraints (exact 5,000 rows, exact `battle_id` alignment with `sample_submission.csv`, binary `{0, 1}` values in `player_won`, zero `NaN`s) and inspect their pairwise prediction agreement.

In [11]:
summary_rows = []
for sub_name, df_sub, cv_acc, cv_std, roc_auc in [
    ("submission_1_compact_top140_xgb.csv", sub1_df, cv_results_map["model4_compact_top140_te"].cv_mean_acc, cv_results_map["model4_compact_top140_te"].cv_std_acc, cv_results_map["model4_compact_top140_te"].oof_roc_auc),
    ("submission_2_conservative_depth4_xgb.csv", sub2_df, cv_results_map["model2_conservative_depth4_pruned_te"].cv_mean_acc, cv_results_map["model2_conservative_depth4_pruned_te"].cv_std_acc, cv_results_map["model2_conservative_depth4_pruned_te"].oof_roc_auc),
    ("submission_3_diverse_xgb_ensemble.csv", sub3_df, float(np.mean(ens_fold_accs)), float(np.std(ens_fold_accs)), float(roc_auc_score(y, ens_oof_probs))),
]:
    v = validate_submission_df(df_sub, sample_sub)
    summary_rows.append({
        "Submission File": sub_name,
        "Valid": v["valid"],
        "Rows": v["n_rows"],
        "CV Accuracy": f"{cv_acc:.4f} +/- {cv_std:.4f}",
        "OOF ROC-AUC": f"{roc_auc:.4f}",
        "Test P1 Win Rate": f"{v['positive_rate']:.4f}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

agree_12 = float(np.mean(sub1_df["player_won"] == sub2_df["player_won"]))
agree_13 = float(np.mean(sub1_df["player_won"] == sub3_df["player_won"]))
agree_23 = float(np.mean(sub2_df["player_won"] == sub3_df["player_won"]))
print(f"\nPairwise Test Prediction Agreement: (Sub1, Sub2)={agree_12:.4f} | (Sub1, Sub3)={agree_13:.4f} | (Sub2, Sub3)={agree_23:.4f}")
print(f"Total Notebook Execution Time: {time.perf_counter() - t_start:.2f}s")

                         Submission File  Valid  Rows       CV Accuracy OOF ROC-AUC Test P1 Win Rate
     submission_1_compact_top140_xgb.csv   True  5000 0.7020 +/- 0.0036      0.7752           0.5202
submission_2_conservative_depth4_xgb.csv   True  5000 0.7008 +/- 0.0024      0.7748           0.5194
   submission_3_diverse_xgb_ensemble.csv   True  5000 0.7030 +/- 0.0017      0.7767           0.5208

Pairwise Test Prediction Agreement: (Sub1, Sub2)=0.9532 | (Sub1, Sub3)=0.9750 | (Sub2, Sub3)=0.9750
Total Notebook Execution Time: 40.22s
